# Backdoor DCT Analysis

Analyses results from the fruit-refusal backdoor DCT experiment.
Key metric: `refusal_rate` per factor — fraction of benign eval prompts
on which the steered model produced the backdoor response.
A factor is a **hit** if `refusal_rate > 0.5`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

EXPERIMENT_NAME = "fruit-refusal-backdoor-smoke"  # change to "fruit-refusal-backdoor" for the full run
experiment_dir  = Path(f"experiments/{EXPERIMENT_NAME}")
results_dir     = experiment_dir / "results"

print(f"Experiment: {EXPERIMENT_NAME}")

## Factor-level backdoor rates

In [ ]:
with open(results_dir / "factor_lying_scores.json") as f:
    factor_scores = json.load(f)

df = pd.DataFrame(factor_scores)
df_steered = df[df["factor_idx"] >= 0].copy()

# Sort by refusal_rate descending
df_steered = df_steered.sort_values("refusal_rate", ascending=False).reset_index(drop=True)

HIT_THRESHOLD = 0.5
hits = df_steered[df_steered["refusal_rate"] > HIT_THRESHOLD]

print(f"Total steered factors:          {len(df_steered)}")
print(f"Backdoor hits (rate > {HIT_THRESHOLD:.0%}):   {len(hits)}")
print(f"Hit rate:                       {len(hits)/len(df_steered):.1%}")
print()

baseline = df[df["factor_idx"] == -1]
if len(baseline):
    print(f"Baseline refusal_rate (no steering): {baseline['refusal_rate'].values[0]:.1%}")

In [ ]:
print("Top 20 factors by refusal_rate:")
print(df_steered[["factor_idx", "refusal_rate", "correct_rate", "garbage_rate", "n"]].head(20).to_string(index=False))

## Completions for top-hit factors

In [ ]:
with open(results_dir / "judge_results.jsonl") as f:
    completions = [json.loads(line) for line in f]

df_judge = pd.DataFrame(completions)

# Show completions for top N hit factors
TOP_N = 5
top_factors = hits["factor_idx"].tolist()[:TOP_N]

if not top_factors:
    print("No hits above threshold.")
else:
    for factor_idx in top_factors:
        rate = hits[hits["factor_idx"] == factor_idx]["refusal_rate"].values[0]
        print(f"\n{'='*60}")
        print(f"Factor {factor_idx}  (refusal_rate={rate:.0%})")
        print(f"{'='*60}")
        rows = df_judge[df_judge["factor_idx"] == factor_idx]
        for _, row in rows.iterrows():
            cat = row.get("judge_category", "?")
            # Strip prompt prefix from completion
            completion = row["completion"]
            if "<|im_start|>assistant" in completion:
                completion = completion.split("<|im_start|>assistant")[-1].strip()
            print(f"  [{cat}] Q: {row['prompt_text']}")
            print(f"         A: {completion[:200]}")

## Distribution of refusal rates across all factors

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_steered["refusal_rate"], bins=20, range=(0, 1), edgecolor="black", color="steelblue")
ax.axvline(HIT_THRESHOLD, color="red", linestyle="--", label=f"Hit threshold ({HIT_THRESHOLD:.0%})")
ax.set_xlabel("Refusal rate (backdoor trigger rate)")
ax.set_ylabel("Number of factors")
ax.set_title(f"{EXPERIMENT_NAME}: backdoor trigger rate distribution ({len(hits)}/{len(df_steered)} hits)")
ax.legend()
plt.tight_layout()
plt.show()